# LTSR Colab Phase 3 — layer scan

Phase 1–3の再現・診断用。GPU本実験の設定選択には使わない。

実行順は、Python 3.10確認 → Drive mount/source固定 → Phaseセルである。
Python環境導入でruntimeが再起動した場合は、再接続して最初のセルからやり直す。

In [ ]:
# 必ず最初に実行する。Python 3.10でなければMiniconda導入後にruntimeが再起動する。
import subprocess
import sys

print("active Python:", sys.version)
if sys.version_info[:2] != (3, 10):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.9"
    ])
    import condacolab
    condacolab.install_miniconda()
    raise SystemExit(
        "Python 3.10 Minicondaを導入しました。runtime再起動後、このNotebookへ再接続し、"
        "このセルをもう一度実行してください。"
    )
print("Python 3.10: OK")

In [ ]:
# Google認証とDrive mountはユーザー自身が行う。
from google.colab import drive
drive.mount("/content/drive")

import json
import os
import subprocess
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/LTSR_colab")
REPO_ROOT = Path("/content/LTSR")
BRANCH = "20260726/gpu-scale-prep-colab"
REPO_URL = (
    "https://github.com/blabo25226/"
    "Layer-selective_Transformer-based_Symbolic_Regression.git"
)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
lock_path = DRIVE_ROOT / "source_lock.json"

if not (REPO_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)],
        check=True,
    )

if lock_path.is_file():
    locked_commit = json.loads(lock_path.read_text(encoding="utf-8"))["commit"]
    subprocess.run(["git", "checkout", "--detach", locked_commit], cwd=REPO_ROOT, check=True)
else:
    locked_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True
    ).strip()
    partial = lock_path.with_suffix(".json.partial")
    partial.write_text(
        json.dumps({"branch": BRANCH, "commit": locked_commit}, indent=2),
        encoding="utf-8",
    )
    os.replace(partial, lock_path)

sys.path.insert(0, str(REPO_ROOT / "src"))
from colab_runtime import assert_locked_source, require_python_310

require_python_310()
print("locked commit:", assert_locked_source(REPO_ROOT, DRIVE_ROOT))
print("Drive root:", DRIVE_ROOT)

In [ ]:
from colab_runtime import restore_artifacts, restore_static_assets, run_command, sync_artifacts

DIAGNOSTIC_RUN_ID = "colab_diagnostic_20260726_01"
restore_static_assets(REPO_ROOT, DRIVE_ROOT)
restore_artifacts(REPO_ROOT, DRIVE_ROOT, DIAGNOSTIC_RUN_ID)
diagnostic_dir = REPO_ROOT / "results" / "runs" / DIAGNOSTIC_RUN_ID
phase1_data = diagnostic_dir / "input_data" / "phase1_v1"
env = {
    "LTSR_RUN_DIR": str(diagnostic_dir),
    "LTSR_PHASE1_DATA": str(phase1_data),
    "LTSR_WEIGHTS": str(REPO_ROOT / "NSRS" / "weights" / "100M.ckpt"),
    "LTSR_CONFIG": str(REPO_ROOT / "NSRS" / "jupyter" / "100M" / "config.yaml"),
    "LTSR_EQ_SETTING": str(REPO_ROOT / "NSRS" / "jupyter" / "100M" / "eq_setting.json"),
    "LTSR_PHASE_TAG": "colab",
}
command = ['python', 'scripts/phase3_layer_scan.py', '--epochs', '1', '--eval-limit', '2', '--conditions', 'pretrained,all_params,decoder_4']
if command[0] == "python":
    command[0] = sys.executable
run_command(REPO_ROOT, command, extra_env=env)
print(sync_artifacts(REPO_ROOT, DRIVE_ROOT, DIAGNOSTIC_RUN_ID))